# PICM — Free GPU Video & AI Upscale (optional external workbench)
This notebook is a **separate optional runtime**, not the production PICM VPS, and **does not automatically connect to mojplaywin.com**.

**Purpose:** animate subjects in a user-uploaded picture with an open-weight image-to-video model, or enhance a user-uploaded picture with Real-ESRGAN. Both are **experimental** and need a sufficiently large GPU. A free Google Colab GPU is **not guaranteed**. **Do not choose paid GPU plans** if you need a zero-cost run.

**Privacy:** The photo is uploaded directly to the notebook session under your Google account, not GitHub or the public PICM site. Review Colab's policies before submitting sensitive faces. Output remains in the notebook runtime until you download it; runtime files can expire.

**Truthfulness:** This produces actual generated frame-to-frame subject motion **when successful**, not a pan/zoom slideshow. A Real-ESRGAN 8K-sized output is **upscaled**, not native 8K capture or newly verified original detail. Motion may distort faces/hands.

**Steps:** Runtime → Change runtime type → choose an available GPU → Run installation → upload your photo → run either video or upscale cell → download results. If Colab offers only paid access or no GPU, stop.


In [ ]:
import torch, os, shutil, subprocess, pathlib, time
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No free GPU available in this Colab session. Stop here; do not buy access.")
print("GPU:", torch.cuda.get_device_name(0), "Memory GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
print("Temporary disk available GB:", round(shutil.disk_usage('/content').free/2**30, 1))
os.makedirs('/content/picm-workspace',exist_ok=True)


In [ ]:
# Install inference dependencies for the open-weight LTX-Video model.
# Downloads will require considerable temporary disk space, typically multiple GB.
%pip -q install -U "diffusers>=0.34" "transformers>=4.44" accelerate safetensors imageio imageio-ffmpeg sentencepiece pillow
print("Dependencies installed. Restart runtime only if Colab asks.")


In [ ]:
from google.colab import files
from PIL import Image, ImageOps
from pathlib import Path
import io
uploaded=files.upload()
if len(uploaded)!=1:
    raise ValueError("Upload exactly one JPG or PNG picture for a single PICM run.")
name,payload=next(iter(uploaded.items()))
if len(payload)>20*1024*1024:
    raise ValueError("Maximum 20 MB input image for this experimental notebook.")
source_image=ImageOps.exif_transpose(Image.open(io.BytesIO(payload))).convert('RGB')
if min(source_image.size)<128:
    raise ValueError("Input too small: minimum side is 128 pixels.")
source_path=Path('/content/picm-workspace/reference.png')
source_image.save(source_path)
print("Reference stored only in Colab temporary runtime:",source_path,source_image.size)


## A. True image-to-video — experimental LTX open-weights
Use this cell when you need **people to move**, not the camera to pan across a photo.

The model may be too large for a free T4 session, fail due to VRAM/disk limits, or change identity. The 49-frame default at 8 fps generates ~6 seconds; 81 frames at 8 fps targets ~10 seconds but needs more resources. For a reliable first attempt, use 49 frames. Models downloaded here are not retained on the PICM server.


In [ ]:
import torch, gc, os
from PIL import ImageOps
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video

# Describe actual human movement. No slideshow, pan/zoom-only or cuts.
PROMPT = ("A single continuous photorealistic shot of the two adults in the "
          "reference photo. Their faces and clothing stay recognizable. "
          "They naturally blink, turn their heads toward each other, smile, "
          "and gently move their hands. Real articulated person movement, "
          "steady camera, believable daylight, no text, no cuts.")
NEGATIVE = "still photo, slideshow, zoom effect, deformed hands, face morphing, duplicate people, flicker, camera cuts"

# Keep width/height multiples of 32 for this model.
WIDTH, HEIGHT = 512, 320
FRAMES = 49  # change to 81 only if you have enough GPU memory and compute
FPS = 8
STEPS = 12

if not torch.cuda.is_available():
    raise RuntimeError("A GPU is required.")
if FRAMES not in (49, 81):
    raise ValueError("Choose 49 or 81 frames, not an unbounded request.")
if shutil.disk_usage('/content').free < 9 * 2**30:
    raise RuntimeError("Not enough free temporary disk for the video model.")
print("Loading LTX image-to-video model. Large download; no charge is requested by this notebook.")
pipe = LTXImageToVideoPipeline.from_pretrained(
    "Lightricks/LTX-Video", torch_dtype=torch.float16
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()
frame=ImageOps.fit(source_image, (WIDTH,HEIGHT), method=Image.Resampling.LANCZOS)
try:
    with torch.inference_mode():
        output=pipe(
            image=frame,
            prompt=PROMPT,
            negative_prompt=NEGATIVE,
            width=WIDTH, height=HEIGHT,
            num_frames=FRAMES,
            num_inference_steps=STEPS,
        ).frames[0]
    video_file="/content/picm-workspace/picm_subject_motion.mp4"
    export_to_video(output,video_file,fps=FPS)
    assert os.path.getsize(video_file)>1024, "Video encoder returned an empty file."
    print("Real AI video generated:",video_file,
          "approximate duration seconds:",round(len(output)/FPS,2))
except torch.cuda.OutOfMemoryError as e:
    raise RuntimeError("Free GPU VRAM insufficient. Restart runtime and try FRAMES=49 or smaller image; do not switch to a paid plan.") from e
finally:
    del pipe
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
from google.colab import files
from pathlib import Path
output=Path("/content/picm-workspace/picm_subject_motion.mp4")
if not output.is_file() or output.stat().st_size<1024:
    raise RuntimeError("No AI video produced. Do not share a false download link.")
files.download(str(output))


## B. AI image upscaling — Real-ESRGAN, **not native 8K**
Runs independently of video. For best results use a good original photo rather than the 256-pixel PICM demo image. First a neural 4× enhancement is applied, then optional conventional Lanczos resizing to a maximum 7680-pixel wide or 4320-pixel high canvas. Upscaling cannot recover guaranteed authentic facial detail.

Source: https://github.com/xinntao/Real-ESRGAN . Model weights are downloaded from the publisher's release; no private image or prompt is uploaded to GitHub.


In [ ]:
# Install Real-ESRGAN only when you want AI photo enhancement.
# Package compatibility can change in Colab. Fail visibly rather than claiming a false success.
import os, subprocess, pathlib
repo=pathlib.Path("/content/picm-workspace/Real-ESRGAN")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","https://github.com/xinntao/Real-ESRGAN.git",str(repo)],check=True)
subprocess.run(["python","-m","pip","install","-q","basicsr","facexlib","gfpgan","realesrgan"],check=True)
print("Real-ESRGAN installed.")


In [ ]:
import os, subprocess
from pathlib import Path
from PIL import Image
weights=repo/"weights"
weights.mkdir(parents=True,exist_ok=True)
# Official, published RealESRGAN_x4plus weights (photo-oriented).
model_path=weights/"RealESRGAN_x4plus.pth"
if not model_path.is_file():
    subprocess.run(["wget","-q","-O",str(model_path),
      "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth"],check=True)
if model_path.stat().st_size < 1_000_000:
    raise RuntimeError("Real-ESRGAN checkpoint download incomplete.")
out_dir=Path('/content/picm-workspace/upscaled')
out_dir.mkdir(exist_ok=True)
args=["python",str(repo/"inference_realesrgan.py"),
      "-n","RealESRGAN_x4plus",
      "-i",str(source_path),"-o",str(out_dir),
      "--model_path",str(model_path),"--tile","128",
      "--outscale","4","--suffix","ai4x","--ext","png"]
subprocess.run(args,check=True)
generated=out_dir/"reference_ai4x.png"
if not generated.is_file():
    raise RuntimeError("Neural upscale did not produce an image.")
assert Image.open(generated).width>source_image.width
print("AI upscaled image:",generated,Image.open(generated).size)


In [ ]:
# Optional 8K-sized *upscaled* export. This is NOT native 8K quality.
from PIL import Image, ImageOps
import math
ai=Image.open(generated).convert('RGB')
# Fit inside an 8K UHD canvas, maintaining aspect ratio. Do not enlarge beyond desired bounds.
ratio=min(7680/ai.width,4320/ai.height)
newsize=(max(1,round(ai.width*ratio)),max(1,round(ai.height*ratio)))
# Using Lanczos after model enhancement is explicitly a conventional resize.
ai=ai.resize(newsize,Image.Resampling.LANCZOS)
p=Path("/content/picm-workspace/picm_ai_upscaled_8k_sized.png")
ai.save(p,optimize=True)
print("Upscaled dimensions:",ai.size,"File:",p,"Bytes:",p.stat().st_size)
from google.colab import files
files.download(str(p))


## Safety and operational limits
- Only upload images you have permission to process.
- This is a **manual, temporary Colab workbench**. No server integrations, hidden paid providers, stored credentials or automated uploads to PICM are included.
- Colab free GPU availability, disk and inference compatibility are not guaranteed. Never claim success until a real MP4 or PNG file was created.
- A 10-second video may take much longer to generate; try a shorter sample first.
- To integrate paid/private GPUs into PICM later, require explicit approval, isolated worker auth, storage encryption, backup, cost cap, and true user download smoke tests.

